# Rebuild Hippo inputs from the eight regenerated sublibraries

Run All on a high-memory Sherlock compute node in the `sc-dl` environment. The inputs are the eight
`/scratch/users/edtng/Hippo_timecourse/Hippo:eTN8_GEX_sub{1..8}/kb_all_main_raw/counts_filtered/adata.h5ad`
files. The older Oak combined H5AD is not used. This notebook rebuilds the production combined query,
recreates the selected `top_specific_200` fixed-reference cNMF projection, validates both, updates the
local path overrides, and executes the current Figure 3 and Figure 4 notebooks.


In [ ]:
from pathlib import Path
import sys

def _find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "cc_perturbseq").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository checkout.")

REPO_ROOT = _find_repo_root()
for path in (REPO_ROOT, REPO_ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print("Repository:", REPO_ROOT)


In [ ]:
from datetime import datetime, timezone
import json
import tempfile
import pandas as pd
from IPython.display import display

from src.cc_perturbseq.figure_notebook import get_path, load_paths
from src.cc_perturbseq.hippo_rebuild import (
    preflight_sublibraries, rebuild_combined, validate_combined,
)
from src.cc_perturbseq.hippo_projection import (
    project_top_specific_200, read_spectra, validate_projected,
)
from src.cc_perturbseq.hippo_workflow import (
    check_runtime, execute_figure_notebooks, run_main_preflight,
    validate_hippo_panel_inputs, write_hippo_overrides,
)

config_file, paths = load_paths(REPO_ROOT)
INPUT_ROOT = get_path(paths, "hippo_rebuild.input_root")
REBUILD_ROOT = get_path(paths, "hippo_rebuild.output_root")
SPECTRA_TPM = get_path(paths, "hippo_rebuild.gene_spectra_tpm")

STRICT_HISTORICAL = True
REBUILD_STALE_OUTPUTS = True
FORCE_REBUILD_ALL = False
COMPRESSION = None
RUN_FIGURE_NOTEBOOKS = True
RUN_FULL_MAIN_PREFLIGHT = True

QUERY_H5AD = REBUILD_ROOT / "Hippo_timecourse_combined.h5ad"
PROJECTED_H5AD = (
    REBUILD_ROOT / "perturbseq_cNMF_projection" / "reprojection_gene_set_comparison"
    / "Hippo_timecourse_reprojected_cNMF_compare_gene_sets.cell_x_program.h5ad"
)
RUN_DIR = REBUILD_ROOT / "runs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")

def build_or_replace_stale(builder, *args, label, **kwargs):
    try:
        return builder(*args, overwrite=FORCE_REBUILD_ALL, **kwargs)
    except FileExistsError as exc:
        if FORCE_REBUILD_ALL or not REBUILD_STALE_OUTPUTS:
            raise
        print(f"{label} is stale or untrusted: {exc}\nRebuilding from current inputs.", flush=True)
        return builder(*args, overwrite=True, **kwargs)

display({
    "config": str(config_file),
    "input_root": str(INPUT_ROOT),
    "combined_output": str(QUERY_H5AD),
    "projection_output": str(PROJECTED_H5AD),
    "run_dir": str(RUN_DIR),
})


In [ ]:
versions = check_runtime(include_figures=RUN_FIGURE_NOTEBOOKS)
inputs = preflight_sublibraries(INPUT_ROOT)
if [x["sublibID"] for x in inputs] != [f"s{i}" for i in range(1, 9)]:
    raise ValueError("Expected the ordered s1...s8 sublibraries.")
table = pd.DataFrame(inputs)
table["mtime_utc"] = pd.to_datetime(table["mtime_ns"], unit="ns", utc=True)
display(table[["sublibID", "path", "size_bytes", "mtime_utc"]])

reference = read_spectra(SPECTRA_TPM, strict_historical=STRICT_HISTORICAL)
print("Reference shape:", reference.shape)
del reference

REBUILD_ROOT.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryFile(dir=REBUILD_ROOT) as probe:
    probe.write(b"write check")
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "settings.json").write_text(json.dumps({
    "inputs": inputs,
    "spectra_tpm": str(SPECTRA_TPM),
    "query_h5ad": str(QUERY_H5AD),
    "projected_h5ad": str(PROJECTED_H5AD),
    "strict_historical": STRICT_HISTORICAL,
    "rebuild_stale_outputs": REBUILD_STALE_OUTPUTS,
    "force_rebuild_all": FORCE_REBUILD_ALL,
    "versions": versions,
}, indent=2) + "\n")

combined = build_or_replace_stale(
    rebuild_combined, INPUT_ROOT, QUERY_H5AD,
    label="Combined Hippo query",
    strict_historical=STRICT_HISTORICAL,
    compression=COMPRESSION,
)
combined["validation_after_final_path"] = validate_combined(
    QUERY_H5AD, strict_historical=STRICT_HISTORICAL,
)
(RUN_DIR / "combined_stage.json").write_text(json.dumps(combined, indent=2) + "\n")
display(combined["validation_after_final_path"])


In [ ]:
projection = build_or_replace_stale(
    project_top_specific_200, QUERY_H5AD, SPECTRA_TPM, PROJECTED_H5AD,
    label="top_specific_200 Hippo projection",
    strict_historical=STRICT_HISTORICAL,
    compression=COMPRESSION,
)
pair_validation = validate_projected(
    QUERY_H5AD, PROJECTED_H5AD, strict_historical=STRICT_HISTORICAL,
)
panel_validation = validate_hippo_panel_inputs(QUERY_H5AD)
config_update = write_hippo_overrides(
    REPO_ROOT, QUERY_H5AD, PROJECTED_H5AD,
    backup_dir=RUN_DIR / "config_backups",
)

for name, payload in {
    "projection_stage.json": projection,
    "pair_validation.json": pair_validation,
    "panel_validation.json": panel_validation,
    "configuration_update.json": config_update,
}.items():
    (RUN_DIR / name).write_text(json.dumps(payload, indent=2) + "\n")

display(projection.get("diagnostics", {}))
display(pair_validation)
display(panel_validation)
display(config_update)


In [ ]:
if RUN_FIGURE_NOTEBOOKS:
    if RUN_FULL_MAIN_PREFLIGHT:
        display(run_main_preflight(REPO_ROOT, RUN_DIR / "main_figure_preflight.log"))
    else:
        print("Full main-figure preflight skipped by explicit setting.")
    display(execute_figure_notebooks(
        REPO_ROOT,
        RUN_DIR,
        notebook_names=("Figure_03.ipynb", "Figure_04.ipynb"),
    ))
else:
    print("Validated Hippo inputs are configured; figure execution is disabled.")

print("Combined query:", QUERY_H5AD)
print("Selected projection:", PROJECTED_H5AD)
print("Run records:", RUN_DIR)
